# Chia PKU theo augmentation group (70/15/15)
**Mục tiêu:** ảnh tăng cường `_1`, `_2`, `_3` theo sau tên gốc phải chung split. Ví dụ `01_missing_hole_01_2.jpg` → `01_missing_hole_01`. Group ở phase này **khác** group tiền tố board của audit tuần 3. **Input:** ảnh/nhãn PKU đã chuẩn bị bởi notebook 08, `configs/base.yaml` nếu có. **Output khi chạy:** `data/splits/{train,val,test}.txt` và `split_manifest.json`. Đây là split mới, không sửa split gốc trong raw. Seed mặc định 42 khi config chưa tồn tại. Không dùng DeepPCB.

In [5]:
from pathlib import Path
from collections import defaultdict, Counter
from IPython.display import display, Markdown
import hashlib, json, random, re
import pandas as pd
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'data/raw/PKU-Market-PCB(Data enhanced version)/Class.txt').is_file()), None)
if ROOT is None: raise FileNotFoundError('Chạy trong project PKU')
RAW = ROOT / 'data/raw/PKU-Market-PCB(Data enhanced version)'
config = ROOT / 'configs/base.yaml'
if config.is_file():
    try: import yaml
    except ImportError as exc: raise ImportError('Cài PyYAML để đọc configs/base.yaml') from exc
    cfg = yaml.safe_load(config.read_text(encoding='utf-8')) or {}
    if not isinstance(cfg, dict): raise ValueError('configs/base.yaml phải là YAML mapping')
    seed = cfg.get('seed', (cfg.get('training') or {}).get('seed', 42))
    seed_source = 'configs/base.yaml' if ('seed' in cfg or 'seed' in (cfg.get('training') or {})) else 'fallback 42 (config không có seed)'
else:
    seed, seed_source = 42, 'fallback 42 (configs/base.yaml chưa tồn tại)'
if isinstance(seed, bool) or not isinstance(seed, int): raise ValueError('seed phải là số nguyên')
display(Markdown(f'**Seed:** {seed} — {seed_source}'))


**Seed:** 42 — fallback 42 (configs/base.yaml chưa tồn tại)

In [6]:
# Chỉ chấp nhận đúng cặp ảnh/label tương ứng trong processed và raw.
exts = {'.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff', '.webp'}
items, missing = [], []
for original_split in ('train', 'valid', 'test'):
    source_images = sorted(p for p in (RAW / original_split / 'images').iterdir() if p.suffix.lower() in exts and p.is_file())
    source_labels = {p.stem: p for p in (RAW / original_split / 'labels').glob('*.txt')}
    if len(source_labels) != len(source_images): missing.append(f'{original_split}: lệch số file ảnh/label gốc')
    for source in source_images:
        image = ROOT / 'data/processed/images' / original_split / source.name
        label = ROOT / 'data/processed/labels' / original_split / (source.stem + '.txt')
        if source.stem not in source_labels or not image.is_symlink() or image.resolve() != source.resolve() or not label.is_file():
            missing.append(str(source.relative_to(ROOT)))
            continue
        group_id = re.sub(r'_[123]$', '', source.stem)
        items.append({'image': str(image.relative_to(ROOT)), 'label': str(label.relative_to(ROOT)), 'source_image': str(source.relative_to(ROOT)), 'source_split': original_split, 'group_id': group_id})
if missing:
    display(pd.DataFrame({'missing_or_invalid': missing[:50]}))
    raise RuntimeError(f'FAIL: {len(missing)} ảnh/nhãn thiếu hoặc symlink sai; chạy notebook 08 trước.')
if len({x['image'] for x in items}) != len(items): raise ValueError('Trùng đường dẫn ảnh processed')
groups = defaultdict(list)
for item in items: groups[item['group_id']].append(item)
if len(groups) < 3: raise ValueError('Cần ít nhất ba group để tạo ba split')
display(Markdown(f'**Đầu vào đã xác nhận:** {len(items)} ảnh PKU, {len(groups)} group; mỗi ảnh có label và symlink đúng nguồn.'))
display(pd.Series(Counter(map(len, groups.values())), name='groups').rename_axis('images_per_group').reset_index())
display(Markdown('Bảng cho biết số group theo số ảnh/group; ảnh tăng cường cùng ảnh gốc là một group.'))


**Đầu vào đã xác nhận:** 3504 ảnh PKU, 693 group; mỗi ảnh có label và symlink đúng nguồn.

,images_per_group,groups
0,6,366
1,4,327


Bảng cho biết số group theo số ảnh/group; ảnh tăng cường cùng ảnh gốc là một group.

In [7]:
# Ưu tiên gần tỷ lệ ảnh 70/15/15 nhưng không được tách group.
fractions = {'train': .70, 'val': .15, 'test': .15}
targets = {s: len(items) * v for s, v in fractions.items()}
rng = random.Random(seed)
ordered = list(groups)
rng.shuffle(ordered)
ordered.sort(key=lambda g: -len(groups[g]))  # shuffle trước sort: phá hòa tái lập được
assigned = {s: [] for s in fractions}
for g in ordered:
    choice = max(fractions, key=lambda s: (targets[s] - len(assigned[s]), -list(fractions).index(s)))
    assigned[choice].extend(groups[g])
if any(not assigned[s] for s in fractions): raise RuntimeError('Không tạo đủ ba split; xem phân bố group')
rows = []
for split, chunk in assigned.items():
    for item in chunk: rows.append({**item, 'split': split})
rows.sort(key=lambda x: (list(fractions).index(x['split']), x['image']))
assert len(rows) == len(items) == len({r['image'] for r in rows})
assert len({r['group_id'] for r in rows}) == sum(len({r['group_id'] for r in assigned[s]}) for s in fractions)
summary = pd.DataFrame([{'split': s, 'images': len(assigned[s]), 'groups': len({r['group_id'] for r in assigned[s]}), 'fraction_images': len(assigned[s])/len(items)} for s in fractions])
display(summary)
display(Markdown('Tỷ lệ là theo **ảnh** (có sai số làm tròn theo group); group không xuất hiện ở hai split. Đây chưa phải bằng chứng không có near-duplicate leakage: phải chạy notebook 10.'))


,split,images,groups,fraction_images
0,train,2452,445,0.699772
1,val,526,124,0.150114
2,test,526,124,0.150114


Tỷ lệ là theo **ảnh** (có sai số làm tròn theo group); group không xuất hiện ở hai split. Đây chưa phải bằng chứng không có near-duplicate leakage: phải chạy notebook 10.

In [8]:
# Ultralytics tìm label bằng thay /images/ thành /labels/ trên đường dẫn ảnh trong txt.
lists = {split: ''.join(str((ROOT / r['image']).absolute()) + '\n' for r in rows if r['split'] == split) for split in fractions}
manifest = {'dataset': 'PKU-Market-PCB(Data enhanced version)', 'seed': seed, 'seed_source': seed_source, 'group_rule': 'strip trailing _1/_2/_3 from filename stem', 'fractions_requested': fractions, 'lists_sha256': {s: hashlib.sha256(text.encode('utf-8')).hexdigest() for s, text in lists.items()}, 'items': rows}
payloads = {ROOT / 'data/splits' / (s + '.txt'): text for s, text in lists.items()}
payloads[ROOT / 'data/splits/split_manifest.json'] = json.dumps(manifest, indent=2, ensure_ascii=False) + '\n'
for path, content in payloads.items():
    if path.exists() and path.read_text(encoding='utf-8') != content: raise FileExistsError(f'Output tồn tại nhưng khác nội dung, kiểm tra trước khi thay: {path}')
for path, content in payloads.items():
    path.parent.mkdir(parents=True, exist_ok=True)
    if not path.exists(): path.write_text(content, encoding='utf-8')
display(Markdown('**Đạt phase 09:** đã lưu `train.txt`, `val.txt`, `test.txt`, `split_manifest.json` vào `data/splits/`. Group cùng ảnh gốc không bị tách; kiểm tra pHash ở phase 10 là bắt buộc.'))


**Đạt phase 09:** đã lưu `train.txt`, `val.txt`, `test.txt`, `split_manifest.json` vào `data/splits/`. Group cùng ảnh gốc không bị tách; kiểm tra pHash ở phase 10 là bắt buộc.

## Kết quả / nhận xét
Các bảng phía trên được tính từ PKU khi chạy. Chỉ đạt điều kiện **group split** nếu mỗi group chỉ thuộc một split và không thiếu ảnh/nhãn; mục tiêu 70/15/15 có thể lệch nhỏ theo kích thước group. Chưa được xem split là an toàn để train trước khi notebook 10 báo PASS.